# NeuralTSP — Dataset Generation & Training Demo

This notebook walks through:
1. Generating a small TSP dataset
2. Inspecting a sample instance
3. Setting up and training the transformer
4. Evaluating on the validation set and plotting predicted tours

In [ ]:
!git clone https://github.com/s1m0n32001/neuraltsp.git
%cd neuraltsp

In [ ]:
import sys, pathlib
from datetime import datetime

sys.path.insert(0, '/content/neuraltsp')
sys.path.insert(0, str(pathlib.Path("").resolve()))

import numpy as np
import torch
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

from neuraltsp.config import DataConfig, ModelConfig
from neuraltsp.data.generator import generate_instance, instances_to_arrays
from neuraltsp.data.dataset import TSPDataset
from neuraltsp.model.model import TSPTransformer
from neuraltsp.model.decode import predict_tour, tour_length
from neuraltsp.train.cache import PathCache
from neuraltsp.train.step import training_step
from neuraltsp.train.eval import evaluate_dataset
from neuraltsp.train.plot import plot_tour

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## 1. Generate Dataset

We use small numbers so the notebook runs quickly. Bump these up for real training.

In [ ]:
DATA_DIR = pathlib.Path("data/demo")
DATA_DIR.mkdir(parents=True, exist_ok=True)

# ── dataset parameters ──────────────────────────────────────────────────────
N_POINTS   = 50      # cities per instance
GRID_SIZE  = 5       # 5×5 = 25 cells
N_TRAIN    = 2_000
N_VAL      = 200
SEED       = 42

def make_split(n, cfg, rng, path):
    instances = [generate_instance(cfg.n_points, cfg.grid_size, rng) for _ in range(n)]
    arrays = instances_to_arrays(instances)
    np.savez_compressed(path,
                        coords=arrays["coords"],
                        cell_ids=arrays["cell_ids"],
                        grid_size=np.int32(cfg.grid_size))
    print(f"  {path}  {arrays['coords'].shape}")

cfg_data = DataConfig(n_points=N_POINTS, grid_size=GRID_SIZE, seed=SEED)
rng_gen  = np.random.default_rng(SEED)

print("Generating...")
make_split(N_TRAIN, cfg_data, rng_gen, DATA_DIR / "train.npz")
make_split(N_VAL,   cfg_data, rng_gen, DATA_DIR / "val.npz")
print("Done.")

## 2. Inspect a Sample Instance

In [ ]:
from scripts.visualize import plot_instance

train_ds = TSPDataset(DATA_DIR / "train.npz")
val_ds   = TSPDataset(DATA_DIR / "val.npz")
print(train_ds)
print(val_ds)

item = train_ds[0]
fig = plot_instance(
    item["coords"].numpy(),
    item["cell_ids"].numpy(),
    grid_size=GRID_SIZE,
    title="Sample training instance (cities coloured by cell)",
)
plt.show()

## 3. Model Setup

In [ ]:
cfg_model = ModelConfig(
    d_model  = 64,
    n_heads  = 4,
    n_layers = 3,
    d_ff     = 128,
    dropout  = 0.0,   # off for this small demo
)

model     = TSPTransformer(cfg_model).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)

n_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {n_params:,}")

## 4. Training

Temperature decays linearly from `T_START` to `T_END` over all epochs.  
Tours are cached per instance and wiped every `CACHE_RESET_EVERY` epochs.

In [ ]:
# ── training hyperparameters ─────────────────────────────────────────────────
N_EPOCHS          = 20
T_START           = 1.0
T_END             = 0.0
CACHE_RESET_EVERY = 5
VAL_EVERY         = 5
N_VAL_PLOTS       = 4    # fixed val instances plotted every VAL_EVERY epochs
RUN_NAME          = None  # set to a string to override the timestamp

# ── per-run log directory ─────────────────────────────────────────────────────
run_name = RUN_NAME or datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
run_dir  = pathlib.Path("logs") / run_name
run_dir.mkdir(parents=True, exist_ok=True)
print(f"Log dir: {run_dir.resolve()}")

# pick a fixed set of val instances to plot (same every epoch)
rng_plot_select = np.random.default_rng(SEED + 1)
val_plot_idxs   = rng_plot_select.choice(len(val_ds), size=min(N_VAL_PLOTS, len(val_ds)), replace=False)

# ── helpers ──────────────────────────────────────────────────────────────────
def temperature_at(epoch, total, t0, t1):
    if total <= 1: return t1
    return t0 + (epoch / (total - 1)) * (t1 - t0)

def save_val_plots(epoch):
    plot_dir = run_dir / "plots" / f"epoch_{epoch:04d}"
    plot_dir.mkdir(parents=True, exist_ok=True)
    rng_dec = np.random.default_rng(0)   # fixed seed → same start city every epoch
    model.eval()
    with torch.no_grad():
        for idx in val_plot_idxs:
            item     = val_ds[int(idx)]
            coords   = item["coords"].numpy()
            cell_ids = item["cell_ids"].numpy()
            tour     = predict_tour(model, coords, cell_ids, GRID_SIZE,
                                    rng=rng_dec, device=device, greedy=True)
            plot_tour(coords, cell_ids, tour, GRID_SIZE,
                      title=f"epoch {epoch}  instance {idx}",
                      path=plot_dir / f"instance_{idx:05d}.png")
    print(f"  plots → {plot_dir}")

# ── training state ────────────────────────────────────────────────────────────
rng   = np.random.default_rng(SEED)
cache = PathCache()

history = {"epoch": [], "loss": [], "accept_rate": [], "val_mean": [], "val_std": []}

import csv
csv_path = run_dir / "val_lengths.csv"
with open(csv_path, "w", newline="") as f:
    csv.writer(f).writerow(["epoch", "mean", "std", "min", "max"])

for epoch in range(N_EPOCHS):
    if epoch % CACHE_RESET_EVERY == 0:
        cache.wipe()

    T     = temperature_at(epoch, N_EPOCHS, T_START, T_END)
    order = rng.permutation(len(train_ds))

    epoch_losses, n_accepted, n_predicted = [], 0, 0

    for idx in tqdm(order, desc=f"Epoch {epoch+1}/{N_EPOCHS}  T={T:.3f}", leave=False):
        idx  = int(idx)
        item = train_ds[idx]

        cached_tour = cache.get(idx)
        if cached_tour is None:
            n_predicted += 1

        result = training_step(
            model, optimizer,
            coords      = item["coords"].numpy(),
            cell_ids    = item["cell_ids"].numpy(),
            grid_size   = GRID_SIZE,
            rng         = rng,
            device      = device,
            temperature = T,
            cached_tour = cached_tour,
        )

        cache.set(idx, result.tour)
        if result.accepted:
            n_accepted += 1
        if result.loss is not None:
            epoch_losses.append(result.loss)

    avg_loss    = float(np.mean(epoch_losses)) if epoch_losses else float("nan")
    accept_rate = n_accepted / len(train_ds)

    history["epoch"].append(epoch + 1)
    history["loss"].append(avg_loss)
    history["accept_rate"].append(accept_rate)

    log = (f"Epoch {epoch+1:3d}  T={T:.3f}  loss={avg_loss:.4f}"
           f"  accept={accept_rate:.1%}  predicted={n_predicted}/{len(train_ds)}")

    # ── validation ────────────────────────────────────────────────────────────
    if (epoch + 1) % VAL_EVERY == 0:
        lengths = evaluate_dataset(model, val_ds, GRID_SIZE, device, seed=0,
                                   show_progress=False)
        history["val_mean"].append(lengths.mean())
        history["val_std"].append(lengths.std())
        log += f"  │  val_len={lengths.mean():.4f} ± {lengths.std():.4f}"

        with open(csv_path, "a", newline="") as f:
            csv.writer(f).writerow([epoch + 1, lengths.mean(), lengths.std(),
                                    lengths.min(), lengths.max()])
        save_val_plots(epoch + 1)
    else:
        history["val_mean"].append(float("nan"))
        history["val_std"].append(float("nan"))

    print(log)

print("Training complete.")

## 5. Training Curves

In [ ]:
epochs      = history["epoch"]
val_epochs  = [e for e, v in zip(epochs, history["val_mean"]) if not np.isnan(v)]
val_means   = [v for v in history["val_mean"] if not np.isnan(v)]
val_stds    = [v for v in history["val_std"]  if not np.isnan(v)]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# loss
axes[0].plot(epochs, history["loss"], marker="o", markersize=3)
axes[0].set_title("Training loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Normalised cross-entropy")
axes[0].grid(True, alpha=0.3)

# acceptance rate
axes[1].plot(epochs, [r * 100 for r in history["accept_rate"]], marker="o",
             markersize=3, color="tab:orange")
axes[1].set_title("SA acceptance rate")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("%")
axes[1].grid(True, alpha=0.3)

# val tour length
axes[2].errorbar(val_epochs, val_means, yerr=val_stds,
                 marker="o", markersize=5, capsize=4, color="tab:green")
axes[2].set_title("Val tour length (greedy)")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("Mean closed-tour length")
axes[2].grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

## 6. Predicted Tours on Validation Instances

Greedy decoding on a few val instances. Blue → red = start → end of tour. ★ marks the start city.

In [ ]:
N_SHOW = 4
rng_plot = np.random.default_rng(0)
plot_idxs = rng_plot.choice(len(val_ds), size=N_SHOW, replace=False)

fig, axes = plt.subplots(1, N_SHOW, figsize=(5 * N_SHOW, 5))

model.eval()
with torch.no_grad():
    for ax, idx in zip(axes, plot_idxs):
        item     = val_ds[int(idx)]
        coords   = item["coords"].numpy()
        cell_ids = item["cell_ids"].numpy()

        tour = predict_tour(model, coords, cell_ids, GRID_SIZE,
                            rng=np.random.default_rng(0), device=device, greedy=True)
        length = tour_length(tour, coords)

        # draw grid
        step = 1.0 / GRID_SIZE
        for k in range(GRID_SIZE + 1):
            ax.axhline(k * step, color="#e0e0e0", lw=0.7)
            ax.axvline(k * step, color="#e0e0e0", lw=0.7)

        # tour edges
        import matplotlib.pyplot as _plt
        cmap_e = _plt.cm.coolwarm
        closed = tour + [tour[0]]
        for s in range(len(tour)):
            a, b = coords[closed[s]], coords[closed[s + 1]]
            ax.annotate("", xy=b, xytext=a,
                        arrowprops=dict(arrowstyle="-|>", color=cmap_e(s / len(tour)),
                                        lw=0.9, mutation_scale=8), zorder=1)

        # cities
        n_cells = GRID_SIZE ** 2
        cmap_c  = _plt.cm.get_cmap("tab20", n_cells)
        ax.scatter(coords[:, 0], coords[:, 1],
                   c=[cmap_c(int(c)) for c in cell_ids], s=25, zorder=2)
        # start star
        ax.scatter(*coords[tour[0]], s=120, c="white", edgecolors="black",
                   linewidths=1.2, marker="*", zorder=3)

        ax.set_xlim(0, 1); ax.set_ylim(0, 1)
        ax.set_aspect("equal")
        ax.set_title(f"val[{idx}]  len={length:.3f}", fontsize=9)
        ax.axis("off")

fig.suptitle("Greedy-decoded val tours after training", y=1.01)
fig.tight_layout()
plt.show()